<a href="https://colab.research.google.com/github/fcoliveira-utfpr/climas_brasil/blob/main/estatisticas_descritivas_climas_brasil.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Estatísticas Descritivas — Classificações Climáticas do Brasil
### Köppen-Geiger (1936) × Camargo (1991) mod. Maluf (2000) × Thornthwaite (1948)
___

## 0. Bibliotecas e configuração
___

In [ ]:
# =========================================================
# BIBLIOTECAS
# =========================================================
!pip install geobr geopandas --quiet

import numpy as np
import pandas as pd
import geopandas as gpd
import geobr
import unicodedata
import warnings
warnings.filterwarnings("ignore")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.1/48.1 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 341.7/341.7 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 49.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 75.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 65.7 MB/s eta 0:00:00


## 1. Carregamento e preparo dos dados
___

### 1.1 Leitura dos CSVs de classificação climática

In [ ]:
urls = {
    "camargo": "https://raw.githubusercontent.com/fcoliveira-utfpr/climas_brasil/refs/heads/main/camargo_municipios_preenchido.csv",
    "koppen":  "https://raw.githubusercontent.com/fcoliveira-utfpr/climas_brasil/refs/heads/main/koppen_municipios_preenchido.csv",
    "thornthwaite": "https://raw.githubusercontent.com/fcoliveira-utfpr/climas_brasil/refs/heads/main/thornthwaite_municipios_preenchido.csv",
}

df_camargo = pd.read_csv(urls["camargo"], sep=",")
df_koppen  = pd.read_csv(urls["koppen"], sep=",")
df_th      = pd.read_csv(urls["thornthwaite"], sep=",")

print("Camargo:", df_camargo.shape)
print("Köppen:", df_koppen.shape)
print("Thornthwaite:", df_th.shape)

Camargo: (5570, 14)
Köppen: (5570, 11)
Thornthwaite: (5570, 16)


### 1.2 Municípios (geobr) e cálculo de área

In [ ]:
def normalize(s):
    if pd.isna(s):
        return ''
    return ''.join(c for c in unicodedata.normalize('NFKD', str(s))
                    if not unicodedata.combining(c)).upper().strip()

UF_NOME = {
    'AC':'Acre','AL':'Alagoas','AP':'Amapá','AM':'Amazonas','BA':'Bahia','CE':'Ceará',
    'DF':'Distrito Federal','ES':'Espírito Santo','GO':'Goiás','MA':'Maranhão','MT':'Mato Grosso',
    'MS':'Mato Grosso do Sul','MG':'Minas Gerais','PA':'Pará','PB':'Paraíba','PR':'Paraná',
    'PE':'Pernambuco','PI':'Piauí','RJ':'Rio de Janeiro','RN':'Rio Grande do Norte',
    'RS':'Rio Grande do Sul','RO':'Rondônia','RR':'Roraima','SC':'Santa Catarina','SP':'São Paulo',
    'SE':'Sergipe','TO':'Tocantins',
}

In [ ]:
municipios_gdf = geobr.read_municipality(code_muni='all', year=2020).to_crs(epsg=4326)
municipios_gdf['municipio_norm'] = municipios_gdf['name_muni'].apply(normalize)
municipios_gdf['uf'] = municipios_gdf['abbrev_state']

# área municipal em km², via reprojeção para EPSG:5880
# (SIRGAS 2000 / Brazil Polyconic — referência equal-area para o território brasileiro)
municipios_gdf['area_km2'] = municipios_gdf.to_crs(epsg=5880).geometry.area / 1e6

print(f'geobr trouxe {len(municipios_gdf)} municípios.')

municipalities_2020_simplified.parquet: 100%|██████████| 20.2M/20.2M [00:00<00:00, 35.5MB/s]


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

geobr trouxe 5570 municípios.


### 1.3 Consolidação em uma única tabela (merge)

Os três CSVs compartilham a chave `code_muni`. Algumas colunas contínuas (temperatura, precipitação, deficiência hídrica etc.) se repetem entre sistemas com o mesmo valor de origem; ao serem mescladas, as versões repetidas do Camargo e do Thornthwaite recebem os sufixos `_cm` e `_th`.

In [ ]:
gdf = municipios_gdf.merge(df_koppen, on='code_muni', how='left')
gdf = gdf.merge(df_camargo, on='code_muni', how='left', suffixes=('', '_cm'))
gdf = gdf.merge(df_th, on='code_muni', how='left', suffixes=('', '_th'))

print('Municípios com Köppen:      ', gdf['classe_koppen_codigo'].notna().sum(), '/', len(gdf))
print('Municípios com Camargo:     ', gdf['cm_termico_codigo'].notna().sum(), '/', len(gdf))
print('Municípios com Thornthwaite:', gdf['th_classe_codigo'].notna().sum(), '/', len(gdf))

gdf.head()

Municípios com Köppen:       5570 / 5570
Municípios com Camargo:      5570 / 5570
Municípios com Thornthwaite: 5570 / 5570


,code_muni,name_muni_x,code_state,abbrev_state_x,name_state,code_region,name_region,year,geometry,municipio_norm,...,th_subtipo,th_pety,th_petr,def_anual_mm_th,excedente_anual_mm_th,pet_anual_mm,indice_umidade_im,indice_hidrico_ih,indice_aridez_ia,preenchido_por_vizinho_th
0,1100015.0,Alta Floresta d'Oeste,11.0,RO,Rondônia,1.0,Norte,2020.0,"POLYGON ((-62.19929 -11.82788, -62.18945 -11.8...",ALTA FLORESTA D'OESTE,...,w,A',a',212.139999,557.433331,1200.913322,35.818516,46.417449,17.664888,False
1,1100023.0,Ariquemes,11.0,RO,Rondônia,1.0,Norte,2020.0,"POLYGON ((-62.53648 -9.73222, -62.52765 -9.736...",ARIQUEMES,...,r,B4',a',152.420000,1189.166675,1122.466667,97.794857,105.942271,13.579022,False
2,1100031.0,Cabixi,11.0,RO,Rondônia,1.0,Norte,2020.0,"POLYGON ((-60.37119 -13.36655, -60.37661 -13.3...",CABIXI,...,w,A',b4',238.443335,600.599997,1283.699997,35.641816,46.786632,18.574693,False
3,1100049.0,Cacoal,11.0,RO,Rondônia,1.0,Norte,2020.0,"POLYGON ((-61.00059 -10.99224, -61.00103 -11.3...",CACOAL,...,w,A',a',195.323335,997.233310,1165.989990,75.475717,85.526747,16.751716,False
4,1100056.0,Cerejeiras,11.0,RO,Rondônia,1.0,Norte,2020.0,"POLYGON ((-61.50149 -13.00501, -61.49426 -13.0...",CEREJEIRAS,...,w,A',b4',260.886671,518.200001,1270.053329,28.476599,40.801436,20.541395,False


## 2. Estatísticas por variável meteorológica
___

### 2.1 Nota sobre os meses de ocorrência

**Não é possível informar em qual mês ocorrem a temperatura mais fria/quente ou a precipitação mais seca.** Os três CSVs de origem trazem apenas os *valores* já agregados (`temp_mes_mais_frio_c`, `temp_mes_mais_quente_c`, `precip_mes_mais_seco_mm`), sem um índice do mês correspondente — essa informação não sobrevive à etapa de extração que gerou os CSVs. Para obter o mês seria necessário reprocessar os 12 valores mensais brutos do TerraClimate diretamente no Google Earth Engine (identificando o mês do mínimo/máximo por município antes de agregar) e gerar uma nova coluna. Consigo montar esse pipeline à parte se for do seu interesse — avise que preparo.

As variáveis não são misturadas entre sistemas — cada tabela usa apenas as variáveis-base do seu próprio sistema. `temp_anual_c` e `temp_mes_mais_frio_c` também compõem a classificação de Camargo, mas como são numericamente idênticas às colunas do Köppen (mesma extração de dados), aparecem só nas tabelas do Köppen, evitando duplicidade.

In [ ]:
def estatisticas_variaveis(gdf_, variaveis, name_col='name_muni', uf_col='abbrev_state'):
    """Para cada (coluna, rótulo, unidade) em `variaveis`, calcula média, desvio padrão,
    valor máximo (+ município/UF onde ocorre) e valor mínimo (+ município/UF onde ocorre)
    em nível nacional. Retorna um DataFrame indexado pelo rótulo da variável."""
    linhas = []
    for col, label, unidade in variaveis:
        serie = gdf_[col]
        idx_max = serie.idxmax()
        idx_min = serie.idxmin()
        linhas.append({
            'variavel': label,
            'unidade': unidade,
            'media': serie.mean(),
            'desvio_padrao': serie.std(),
            'valor_maximo': serie.loc[idx_max],
            'municipio_maximo': f"{gdf_.loc[idx_max, name_col]}/{gdf_.loc[idx_max, uf_col]}",
            'valor_minimo': serie.loc[idx_min],
            'municipio_minimo': f"{gdf_.loc[idx_min, name_col]}/{gdf_.loc[idx_min, uf_col]}",
        })
    df_stats = pd.DataFrame(linhas).set_index('variavel')
    cols_round = ['media', 'desvio_padrao', 'valor_maximo', 'valor_minimo']
    df_stats[cols_round] = df_stats[cols_round].round(2)
    return df_stats


def estatisticas_variaveis_por_uf(gdf_, variaveis, uf_col='abbrev_state', name_col='name_muni'):
    """Mesma lógica de `estatisticas_variaveis`, mas calculada separadamente para cada UF
    (incluindo o DF). Retorna um DataFrame longo indexado por (uf, variavel)."""
    linhas = []
    for uf in sorted(gdf_[uf_col].dropna().unique()):
        sub = gdf_[gdf_[uf_col] == uf]
        for col, label, unidade in variaveis:
            serie = sub[col]
            if serie.notna().sum() == 0:
                continue
            idx_max = serie.idxmax()
            idx_min = serie.idxmin()
            linhas.append({
                'uf': uf,
                'uf_nome': UF_NOME.get(uf, uf),
                'variavel': label,
                'unidade': unidade,
                'n_municipios': int(serie.notna().sum()),
                'media': serie.mean(),
                'desvio_padrao': serie.std(),
                'valor_maximo': serie.loc[idx_max],
                'municipio_maximo': sub.loc[idx_max, name_col],
                'valor_minimo': serie.loc[idx_min],
                'municipio_minimo': sub.loc[idx_min, name_col],
            })
    df_stats = pd.DataFrame(linhas)
    cols_round = ['media', 'desvio_padrao', 'valor_maximo', 'valor_minimo']
    df_stats[cols_round] = df_stats[cols_round].round(2)
    return df_stats.set_index(['uf', 'variavel'])

In [ ]:
variaveis_koppen = [
    ('precip_anual_mm',         'Precipitação Anual',             'mm'),
    ('precip_mes_mais_seco_mm', 'Precipitação do Mês mais Seco',  'mm'),
    ('temp_anual_c',            'Temperatura Média Anual',        '°C'),
    ('temp_mes_mais_frio_c',    'Temperatura do Mês mais Frio',   '°C'),
    ('temp_mes_mais_quente_c',  'Temperatura do Mês mais Quente', '°C'),
]

variaveis_camargo = [
    ('def_anual_mm',       'Deficiência Hídrica Anual', 'mm'),
    ('excedente_anual_mm', 'Excedente Hídrico Anual',   'mm'),
]

variaveis_thornthwaite = [
    ('def_anual_mm_th',       'Deficiência Hídrica Anual',           'mm'),
    ('excedente_anual_mm_th', 'Excedente Hídrico Anual',             'mm'),
    ('pet_anual_mm',          'Evapotranspiração Potencial Anual',   'mm'),
    ('indice_umidade_im',     'Índice de Umidade (Im)',              ''),
    ('indice_hidrico_ih',     'Índice Hídrico (Ih)',                 ''),
    ('indice_aridez_ia',      'Índice de Aridez (Ia)',               ''),
]

### 2.2 Brasil (nível nacional)

#### 2.2.1 Köppen-Geiger (1936)

In [ ]:
df_stats_koppen = estatisticas_variaveis(gdf, variaveis_koppen)
df_stats_koppen

,unidade,media,desvio_padrao,valor_maximo,municipio_maximo,valor_minimo,municipio_minimo
variavel,,,,,,,
Precipitação Anual,mm,1362.73,462.62,3678.73,Maraã/AM,308.40,Campo Formoso/BA
Precipitação do Mês mais Seco,mm,32.71,36.75,220.27,São Gabriel da Cachoeira/AM,0.00,Mateiros/TO
Temperatura Média Anual,°C,22.68,2.93,28.47,São Luís Gonzaga do Maranhão/MA,14.16,Bom Jardim da Serra/SC
Temperatura do Mês mais Frio,°C,19.68,4.15,27.43,Ilha Grande/PI,9.62,Campos do Jordão/SP
Temperatura do Mês mais Quente,°C,25.11,2.20,30.61,Teresina/PI,17.95,Bom Jardim da Serra/SC


#### 2.2.2 Camargo (1991) mod. Maluf (2000)

In [ ]:
df_stats_camargo = estatisticas_variaveis(gdf, variaveis_camargo)
df_stats_camargo

,unidade,media,desvio_padrao,valor_maximo,municipio_maximo,valor_minimo,municipio_minimo
variavel,,,,,,,
Deficiência Hídrica Anual,mm,374.04,277.18,1289.02,Sobradinho/BA,2.01,Japurá/AM
Excedente Hídrico Anual,mm,466.38,335.28,2580.80,Maraã/AM,14.90,Campo Formoso/BA


#### 2.2.3 Thornthwaite (1948)

In [ ]:
df_stats_thornthwaite = estatisticas_variaveis(gdf, variaveis_thornthwaite)
df_stats_thornthwaite

,unidade,media,desvio_padrao,valor_maximo,municipio_maximo,valor_minimo,municipio_minimo
variavel,,,,,,,
Deficiência Hídrica Anual,mm,374.04,277.18,1289.02,Sobradinho/BA,2.01,Japurá/AM
Excedente Hídrico Anual,mm,466.38,335.28,2580.80,Maraã/AM,14.90,Campo Formoso/BA
Evapotranspiração Potencial Anual,mm,1269.88,166.46,1729.81,Grossos/RN,888.51,Urubici/SC
Índice de Umidade (Im),,22.49,38.42,234.17,Maraã/AM,-47.32,Campo Formoso/BA
Índice Hídrico (Ih),,39.05,29.65,234.36,Maraã/AM,0.99,Campo Formoso/BA
Índice de Aridez (Ia),,27.60,17.93,80.51,Campo Formoso/BA,0.19,Japurá/AM


### 2.3 Por Unidade da Federação (UF/DF)

Mesmas estatísticas do item 2.2, calculadas separadamente para cada estado e para o Distrito Federal. O índice é composto por `(uf, variavel)`; use `.loc['PR']`, por exemplo, para ver só o Paraná.

#### 2.3.1 Köppen-Geiger (1936)

In [ ]:
df_stats_koppen_uf = estatisticas_variaveis_por_uf(gdf, variaveis_koppen)
df_stats_koppen_uf

uf_nome unidade  n_municipios    media  \
uf variavel                                                                   
AC Precipitação Anual                   Acre      mm            22  1873.35   
   Precipitação do Mês mais Seco        Acre      mm            22    35.04   
   Temperatura Média Anual              Acre      °C            22    25.34   
   Temperatura do Mês mais Frio         Acre      °C            22    23.95   
   Temperatura do Mês mais Quente       Acre      °C            22    26.11   
...                                      ...     ...           ...      ...   
TO Precipitação Anual              Tocantins      mm           139  1563.01   
   Precipitação do Mês mais Seco   Tocantins      mm           139     3.14   
   Temperatura Média Anual         Tocantins      °C           139    25.84   
   Temperatura do Mês mais Frio    Tocantins      °C           139    24.57   
   Temperatura do Mês mais Quente  Tocantins      °C           139    27.60   

                                   desvio_padrao  valor_maximo  \
uf variavel                                                      
AC Precipitação Anual                     159.18       2256.33   
   Precipitação do Mês mais Seco           11.85         69.67   
   Temperatura Média Anual                  0.37         26.16   
   Temperatura do Mês mais Frio             0.51         24.90   
   Temperatura do Mês mais Quente           0.45         27.13   
...                                          ...           ...   
TO Precipitação Anual                     137.15       1807.60   
   Precipitação do Mês mais Seco            3.62         12.93   
   Temperatura Média Anual                  0.62         27.13   
   Temperatura do Mês mais Frio             0.93         26.15   
   Temperatura do Mês mais Quente           0.53         28.59   

                                             municipio_maximo  valor_minimo  \
uf variavel                                                                   
AC Precipitação Anual                             Mâncio Lima       1633.33   
   Precipitação do Mês mais Seco                  Mâncio Lima         19.13   
   Temperatura Média Anual                         Acrelândia         24.83   
   Temperatura do Mês mais Frio                    Acrelândia         23.05   
   Temperatura do Mês mais Quente                  Acrelândia         25.49   
...                                                       ...           ...   
TO Precipitação Anual                                 Juarina       1200.07   
   Precipitação do Mês mais Seco                  Esperantina          0.00   
   Temperatura Média Anual         São Sebastião do Tocantins         23.63   
   Temperatura do Mês mais Frio    São Sebastião do Tocantins         21.94   
   Temperatura do Mês mais Quente                   Combinado         25.46   

                                       municipio_minimo  
uf variavel                                              
AC Precipitação Anual              Marechal Thaumaturgo  
   Precipitação do Mês mais Seco             Acrelândia  
   Temperatura Média Anual                 Assis Brasil  
   Temperatura do Mês mais Frio            Assis Brasil  
   Temperatura do Mês mais Quente              Tarauacá  
...                                                 ...  
TO Precipitação Anual                          Mateiros  
   Precipitação do Mês mais Seco               Mateiros  
   Temperatura Média Anual               Monte do Carmo  
   Temperatura do Mês mais Frio          Monte do Carmo  
   Temperatura do Mês mais Quente        Monte do Carmo  

[135 rows x 9 columns]

#### 2.3.2 Camargo (1991) mod. Maluf (2000)

In [ ]:
df_stats_camargo_uf = estatisticas_variaveis_por_uf(gdf, variaveis_camargo)
df_stats_camargo_uf

uf_nome unidade  n_municipios  \
uf variavel                                                               
AC Deficiência Hídrica Anual                 Acre      mm            22   
   Excedente Hídrico Anual                   Acre      mm            22   
AL Deficiência Hídrica Anual              Alagoas      mm           102   
   Excedente Hídrico Anual                Alagoas      mm           102   
AM Deficiência Hídrica Anual             Amazonas      mm            62   
   Excedente Hídrico Anual               Amazonas      mm            62   
AP Deficiência Hídrica Anual                Amapá      mm            16   
   Excedente Hídrico Anual                  Amapá      mm            16   
BA Deficiência Hídrica Anual                Bahia      mm           417   
   Excedente Hídrico Anual                  Bahia      mm           417   
CE Deficiência Hídrica Anual                Ceará      mm           184   
   Excedente Hídrico Anual                  Ceará      mm           184   
DF Deficiência Hídrica Anual     Distrito Federal      mm             1   
   Excedente Hídrico Anual       Distrito Federal      mm             1   
ES Deficiência Hídrica Anual       Espírito Santo      mm            78   
   Excedente Hídrico Anual         Espírito Santo      mm            78   
GO Deficiência Hídrica Anual                Goiás      mm           246   
   Excedente Hídrico Anual                  Goiás      mm           246   
MA Deficiência Hídrica Anual             Maranhão      mm           217   
   Excedente Hídrico Anual               Maranhão      mm           217   
MG Deficiência Hídrica Anual         Minas Gerais      mm           853   
   Excedente Hídrico Anual           Minas Gerais      mm           853   
MS Deficiência Hídrica Anual   Mato Grosso do Sul      mm            79   
   Excedente Hídrico Anual     Mato Grosso do Sul      mm            79   
MT Deficiência Hídrica Anual          Mato Grosso      mm           141   
   Excedente Hídrico Anual            Mato Grosso      mm           141   
PA Deficiência Hídrica Anual                 Pará      mm           144   
   Excedente Hídrico Anual                   Pará      mm           144   
PB Deficiência Hídrica Anual              Paraíba      mm           223   
   Excedente Hídrico Anual                Paraíba      mm           223   
PE Deficiência Hídrica Anual           Pernambuco      mm           185   
   Excedente Hídrico Anual             Pernambuco      mm           185   
PI Deficiência Hídrica Anual                Piauí      mm           224   
   Excedente Hídrico Anual                  Piauí      mm           224   
PR Deficiência Hídrica Anual               Paraná      mm           399   
   Excedente Hídrico Anual                 Paraná      mm           399   
RJ Deficiência Hídrica Anual       Rio de Janeiro      mm            92   
   Excedente Hídrico Anual         Rio de Janeiro      mm            92   
RN Deficiência Hídrica Anual  Rio Grande do Norte      mm           167   
   Excedente Hídrico Anual    Rio Grande do Norte      mm           167   
RO Deficiência Hídrica Anual             Rondônia      mm            52   
   Excedente Hídrico Anual               Rondônia      mm            52   
RR Deficiência Hídrica Anual              Roraima      mm            15   
   Excedente Hídrico Anual                Roraima      mm            15   
RS Deficiência Hídrica Anual    Rio Grande do Sul      mm           497   
   Excedente Hídrico Anual      Rio Grande do Sul      mm           497   
SC Deficiência Hídrica Anual       Santa Catarina      mm           295   
   Excedente Hídrico Anual         Santa Catarina      mm           295   
SE Deficiência Hídrica Anual              Sergipe      mm            75   
   Excedente Hídrico Anual                Sergipe      mm            75   
SP Deficiência Hídrica Anual            São Paulo      mm           645   
   Excedente Hídrico Anual              São Paulo      mm          

#### 2.3.3 Thornthwaite (1948)

In [ ]:
df_stats_thornthwaite_uf = estatisticas_variaveis_por_uf(gdf, variaveis_thornthwaite)
df_stats_thornthwaite_uf

uf_nome unidade  n_municipios  \
uf variavel                                                             
AC Deficiência Hídrica Anual               Acre      mm            22   
   Excedente Hídrico Anual                 Acre      mm            22   
   Evapotranspiração Potencial Anual       Acre      mm            22   
   Índice de Umidade (Im)                  Acre                    22   
   Índice Hídrico (Ih)                     Acre                    22   
...                                         ...     ...           ...   
TO Excedente Hídrico Anual            Tocantins      mm           139   
   Evapotranspiração Potencial Anual  Tocantins      mm           139   
   Índice de Umidade (Im)             Tocantins                   139   
   Índice Hídrico (Ih)                Tocantins                   139   
   Índice de Aridez (Ia)              Tocantins                   139   

                                        media  desvio_padrao  valor_maximo  \
uf variavel                                                                  
AC Deficiência Hídrica Anual           103.70          27.22        143.61   
   Excedente Hídrico Anual             903.55         157.15       1252.83   
   Evapotranspiração Potencial Anual  1074.19          30.70       1124.44   
   Índice de Umidade (Im)               78.70          17.74        117.96   
   Índice Hídrico (Ih)                  84.46          16.53        120.21   
...                                       ...            ...           ...   
TO Excedente Hídrico Anual             594.12         132.99        840.67   
   Evapotranspiração Potencial Anual  1365.75          46.95       1509.97   
   Índice de Umidade (Im)               26.49          11.79         46.44   
   Índice Hídrico (Ih)                  43.75          10.62         62.87   
   Índice de Aridez (Ia)                28.76           4.61         39.45   

                                               municipio_maximo  valor_minimo  \
uf variavel                                                                     
AC Deficiência Hídrica Anual                          Brasiléia         39.10   
   Excedente Hídrico Anual                          Mâncio Lima        672.10   
   Evapotranspiração Potencial Anual                 Acrelândia       1018.69   
   Índice de Umidade (Im)                           Mâncio Lima         55.52   
   Índice Hídrico (Ih)                              Mâncio Lima         62.17   
...                                                         ...           ...   
TO Excedente Hídrico Anual                           Araguacema        282.47   
   Evapotranspiração Potencial Anual        Aurora do Tocantins       1287.20   
   Índice de Umidade (Im)             Bandeirantes do Tocantins          0.25   
   Índice Hídrico (Ih)                               Araguacema         20.49   
   Índice de Aridez (Ia)                       Rio da Conceição         21.25   

                                          municipio_minimo  
uf variavel                                                 
AC Deficiência Hídrica Anual                   Mâncio Lima  
   Excedente Hídrico Anual            Marechal Thaumaturgo  
   Evapotranspiração Potencial Anual              Tarauacá  
   Índice de Umidade (Im)             Marechal Thaumaturgo  
   Índice Hídrico (Ih)                Marechal Thaumaturgo  
...                                                    ...  
TO Excedente Hídrico Anual                   Campos Lindos  
   Evapotranspiração Potencial Anual  Paraíso do Tocantins  
   Índice de Umidade (Im)                         Mateiros  
   Índice Hídrico (Ih)                       Campos Lindos  
   Índice de Aridez (Ia)              Santa Fé do Araguaia  

[162 rows x 9 columns]

## 3. Número de municípios e área por classificação climática
___

Contagem de municípios e área (km²) por classe, usando a **classificação completa** de cada sistema (`classe_koppen`, `camargo_completo`, `thornthwaite_completo`). A área (`area_km2`) já foi calculada na seção 1.2, a partir da reprojeção para EPSG:5880.

In [ ]:
def contagem_area_por_classe(gdf_, coluna_classe):
    """Conta municípios e soma a área (km²) por classe climática completa, em nível
    nacional, ordenado do mais frequente para o menos frequente."""
    agg = gdf_.groupby(coluna_classe, dropna=True).agg(
        n_municipios=('code_muni', 'count'),
        area_km2=('area_km2', 'sum'),
    ).sort_values('n_municipios', ascending=False)

    agg['pct_municipios'] = (100 * agg['n_municipios'] / agg['n_municipios'].sum()).round(2)
    agg['pct_area'] = (100 * agg['area_km2'] / agg['area_km2'].sum()).round(2)
    agg['area_km2'] = agg['area_km2'].round(0)

    return agg[['n_municipios', 'pct_municipios', 'area_km2', 'pct_area']]


def contagem_area_por_classe_uf(gdf_, coluna_classe, uf_col='abbrev_state'):
    """Mesma lógica de `contagem_area_por_classe`, mas calculada separadamente para cada UF
    (incluindo o DF). Os percentuais são relativos ao total de cada UF."""
    agg = gdf_.groupby([uf_col, coluna_classe], dropna=True).agg(
        n_municipios=('code_muni', 'count'),
        area_km2=('area_km2', 'sum'),
    )
    agg['pct_municipios_uf'] = (100 * agg['n_municipios'] /
                                 agg.groupby(level=0)['n_municipios'].transform('sum')).round(2)
    agg['pct_area_uf'] = (100 * agg['area_km2'] /
                           agg.groupby(level=0)['area_km2'].transform('sum')).round(2)
    agg['area_km2'] = agg['area_km2'].round(0)

    agg = agg.reset_index().sort_values([uf_col, 'n_municipios'], ascending=[True, False])
    return agg.set_index([uf_col, coluna_classe])[['n_municipios', 'pct_municipios_uf', 'area_km2', 'pct_area_uf']]

### 3.1 Brasil (nível nacional)

#### 3.1.1 Köppen-Geiger (1936)

In [ ]:
df_contagem_koppen = contagem_area_por_classe(gdf, 'classe_koppen')
df_contagem_koppen

,n_municipios,pct_municipios,area_km2,pct_area
classe_koppen,,,,
Aw,2120,38.06,3730547.0,43.41
Cfa,996,17.88,465084.0,5.41
Cwa,558,10.02,233270.0,2.71
As,546,9.80,191034.0,2.22
BSh,468,8.40,517186.0,6.02
Cfb,279,5.01,161800.0,1.88
Am,261,4.69,1692644.0,19.70
Cwb,181,3.25,69156.0,0.80
Af,147,2.64,1513885.0,17.62


#### 3.1.2 Camargo (1991) mod. Maluf (2000)

In [ ]:
df_contagem_camargo = contagem_area_por_classe(gdf, 'camargo_completo')
df_contagem_camargo

,n_municipios,pct_municipios,area_km2,pct_area
camargo_completo,,,,
TR-MOi,1315,23.61,3189222.0,37.11
ST-MOi,938,16.84,425057.0,4.95
TR-SEp,701,12.59,444109.0,5.17
ST-UMp,461,8.28,138503.0,1.61
TR-SEi,406,7.29,671735.0,7.82
TR-MOp,391,7.02,542716.0,6.32
TE-UMp,221,3.97,115019.0,1.34
ST-UMv,202,3.63,73966.0,0.86
ST-UMi,165,2.96,81365.0,0.95


#### 3.1.3 Thornthwaite (1948)

In [ ]:
df_contagem_thornthwaite = contagem_area_por_classe(gdf, 'thornthwaite_completo')
df_contagem_thornthwaite

,n_municipios,pct_municipios,area_km2,pct_area
thornthwaite_completo,,,,
DdA'b3',450,8.08,399954.0,4.65
C2wA'b2',375,6.73,216900.0,2.52
DdA'b2',375,6.73,257176.0,2.99
B1wA'b4',255,4.58,395539.0,4.60
B1wA'b3',199,3.57,172824.0,2.01
...,...,...,...,...
C2sA'b1',1,0.02,335.0,0.00
C2sA'c2',1,0.02,1056.0,0.01
DdB4'b2',1,0.02,873.0,0.01


### 3.2 Por Unidade da Federação (UF/DF)

Índice composto por `(uf, classe)`; use `.loc['PR']`, por exemplo, para ver só o Paraná.

#### 3.2.1 Köppen-Geiger (1936)

In [ ]:
df_contagem_koppen_uf = contagem_area_por_classe_uf(gdf, 'classe_koppen')
df_contagem_koppen_uf

n_municipios  pct_municipios_uf  area_km2  \
abbrev_state classe_koppen                                              
AC           Am                       15              68.18  141214.0   
             Aw                        6              27.27   23926.0   
             Af                        1               4.55    5763.0   
AL           As                       79              77.45   21344.0   
             BSh                      15              14.71    6137.0   
...                                  ...                ...       ...   
SP           Cwb                      46               7.13   11899.0   
             Cfb                      45               6.98   17923.0   
             Af                       11               1.71    6676.0   
             Am                       10               1.55    7334.0   
TO           Aw                      139             100.00  278789.0   

                            pct_area_uf  
abbrev_state classe_koppen               
AC           Am                   82.63  
             Aw                   14.00  
             Af                    3.37  
AL           As                   73.38  
             BSh                  21.10  
...                                 ...  
SP           Cwb                   4.77  
             Cfb                   7.19  
             Af                    2.68  
             Am                    2.94  
TO           Aw                  100.00  

[88 rows x 4 columns]

#### 3.2.2 Camargo (1991) mod. Maluf (2000)

In [ ]:
df_contagem_camargo_uf = contagem_area_por_classe_uf(gdf, 'camargo_completo')
df_contagem_camargo_uf

n_municipios  pct_municipios_uf  area_km2  \
abbrev_state camargo_completo                                              
AC           TR-UMi                      22             100.00  170903.0   
AL           TR-MOv                      42              41.18   11071.0   
             TR-SEp                      30              29.41   10639.0   
             TR-SEv                      22              21.57    5617.0   
             TR-MOp                       6               5.88    1386.0   
...                                     ...                ...       ...   
SP           TE-UMi                      17               2.64    6721.0   
             TR-SEi                       7               1.09    2543.0   
             ST-UMp                       1               0.16     337.0   
             TE-MOi                       1               0.16     160.0   
TO           TR-MOi                     139             100.00  278789.0   

                               pct_area_uf  
abbrev_state camargo_completo               
AC           TR-UMi                 100.00  
AL           TR-MOv                  38.06  
             TR-SEp                  36.58  
             TR-SEv                  19.31  
             TR-MOp                   4.77  
...                                    ...  
SP           TE-UMi                   2.70  
             TR-SEi                   1.02  
             ST-UMp                   0.14  
             TE-MOi                   0.06  
TO           TR-MOi                 100.00  

[120 rows x 4 columns]

#### 3.2.3 Thornthwaite (1948)

In [ ]:
df_contagem_thornthwaite_uf = contagem_area_por_classe_uf(gdf, 'thornthwaite_completo')
df_contagem_thornthwaite_uf

n_municipios  pct_municipios_uf  area_km2  \
abbrev_state thornthwaite_completo                                              
AC           B4rB4'b4'                         9              40.91  114679.0   
             B3rB4'b4'                         7              31.82   32472.0   
             B2rB4'b4'                         4              18.18   16133.0   
             ArB4'b4'                          1               4.55    5763.0   
             B3rB4'a'                          1               4.55    1857.0   
...                                          ...                ...       ...   
TO           C2w2A'b4'                        13               9.35   36592.0   
             C2wA'b4'                          7               5.04   18089.0   
             C2w2A'a'                          4               2.88   12132.0   
             B1w2A'a'                          1               0.72    5171.0   
             B1wA'b4'                          1               0.72    2164.0   

                                    pct_area_uf  
abbrev_state thornthwaite_completo               
AC           B4rB4'b4'                    67.10  
             B3rB4'b4'                    19.00  
             B2rB4'b4'                     9.44  
             ArB4'b4'                      3.37  
             B3rB4'a'                      1.09  
...                                         ...  
TO           C2w2A'b4'                    13.13  
             C2wA'b4'                      6.49  
             C2w2A'a'                      4.35  
             B1w2A'a'                      1.85  
             B1wA'b4'                      0.78  

[346 rows x 4 columns]

## 4. Exportação (opcional)
___

In [ ]:
# Descomente para exportar as tabelas em CSV
# df_stats_koppen.to_csv('estatisticas_koppen_brasil.csv')
# df_stats_camargo.to_csv('estatisticas_camargo_brasil.csv')
# df_stats_thornthwaite.to_csv('estatisticas_thornthwaite_brasil.csv')
# df_stats_koppen_uf.to_csv('estatisticas_koppen_uf.csv')
# df_stats_camargo_uf.to_csv('estatisticas_camargo_uf.csv')
# df_stats_thornthwaite_uf.to_csv('estatisticas_thornthwaite_uf.csv')
# df_contagem_koppen.to_csv('contagem_area_koppen_brasil.csv')
# df_contagem_camargo.to_csv('contagem_area_camargo_brasil.csv')
# df_contagem_thornthwaite.to_csv('contagem_area_thornthwaite_brasil.csv')
# df_contagem_koppen_uf.to_csv('contagem_area_koppen_uf.csv')
# df_contagem_camargo_uf.to_csv('contagem_area_camargo_uf.csv')
# df_contagem_thornthwaite_uf.to_csv('contagem_area_thornthwaite_uf.csv')